# 🏐 Análisis de vídeo de voleibol — pruebas

Este cuaderno usa el mismo código que funcionará después en el MacBook (`video/voley_cv` del repositorio).

**Antes de empezar**
1. Menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**.
2. Sube el vídeo original (H.264, no el proxy) a tu Google Drive, por ejemplo a `Mi unidad/voley/`.
3. Ejecuta las celdas en orden con ▶. Cada una explica qué hace.

Nada sale de tu Drive: el vídeo se procesa en la máquina de Colab y los resultados se guardan en tu carpeta de Drive.

In [ ]:
#@title 1. Preparar: GPU, Google Drive y código
import os, subprocess, sys
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout
print('GPU:', gpu.strip() or 'NINGUNA → Entorno de ejecución → Cambiar tipo → T4 GPU')

from google.colab import drive
drive.mount('/content/drive')

BRANCH = 'claude/volleyball-stats-github-pages-cf7xwk'
if not os.path.exists('/content/voley-app'):
    !git clone -q -b {BRANCH} https://github.com/carero2/voley-app.git /content/voley-app
else:
    !git -C /content/voley-app pull -q
%cd /content/voley-app/video
!pip install -q -r requirements.txt
sys.path.insert(0, '/content/voley-app/video')
print('Listo')

In [ ]:
#@title 2. Tu vídeo
VIDEO = '/content/drive/MyDrive/voley/partido.mov'  #@param {type:"string"}
VISTA = 'lateral'  #@param ["lateral", "fondo"]
CARPETA = '/content/drive/MyDrive/voley/resultados'  #@param {type:"string"}

from voley_cv.video_io import video_info
os.makedirs(CARPETA, exist_ok=True)
info = video_info(VIDEO)
print(f"{info['width']}x{info['height']} · {info['fps']:.2f} fps · {info['duration'] / 60:.1f} min")

## Calibrar el campo
Marca con clics las **4 esquinas del campo de voley** (las líneas exteriores, no las de otros deportes), en este orden, tal como se ven en la imagen:
**1** cerca-izquierda · **2** cerca-derecha · **3** lejos-derecha · **4** lejos-izquierda.

Después, si se ven, marca los **2 extremos de la línea central** (la que queda justo debajo de la red); mejoran la calibración con el gran angular. Si no se ven, pulsa «Saltar este punto». Termina con **Listo**.

Elige un segundo en el que las esquinas no estén tapadas por jugadores.

In [ ]:
#@title 3. Calibrar el campo (clics sobre la imagen)
SEGUNDO = 30  #@param {type:"number"}
import cv2
from google.colab.patches import cv2_imshow
from voley_cv.video_io import read_frame
from voley_cv.calibrate_ui import pick_points_colab
from voley_cv.court import Court

frame = read_frame(VIDEO, SEGUNDO)
corners, extra = pick_points_colab(frame, VISTA)
court = Court.from_clicks(VISTA, corners, (frame.shape[1], frame.shape[0]), extra)
CAMPO = f'{CARPETA}/campo.json'
court.save(CAMPO)
preview = court.draw_overlay(frame)
cv2.imwrite(f'{CARPETA}/campo_comprobacion.jpg', preview)
print(f'Error de reproyección: {court.reprojection_error():.1f} px (con 6 puntos, menos de ~5 px está bien)')
print('Las líneas amarillas deben coincidir con las del campo. Si no, repite esta celda.')
cv2_imshow(cv2.resize(preview, (1280, 720)))

## Analizar un tramo
Detecta jugadores y balón, sigue el balón, busca toques, pasos de red y posesiones, y hace un vídeo anotado.

- **CADA = 2** analiza uno de cada dos fotogramas (el doble de rápido; suficiente para una primera prueba).
- **MOSAICOS**: el balón es pequeño; la imagen se divide en trozos para verlo mejor. 3x2 es un buen punto de partida; 1x1 = sin mosaicos (más rápido, casi no verá el balón).
- **ETIQUETAS** (opcional): el archivo `.json` que descargas de la herramienta de etiquetar puntos, para comparar punto a punto.
- **PESOS** (más adelante): el modelo reentrenado con tus etiquetas.

En una T4, un minuto de vídeo con CADA = 2 tarda unos minutos.

In [ ]:
#@title 4. Analizar un tramo
INICIO = 60  #@param {type:"number"}
FIN = 120  #@param {type:"number"}
CADA = 2  #@param {type:"integer"}
MODELO = 'small'  #@param ["nano", "small", "medium"]
MOSAICOS = '3x2'  #@param ["1x1", "2x1", "3x2", "4x3"]
ETIQUETAS = ''  #@param {type:"string"}
PESOS = ''  #@param {type:"string"}

from voley_cv.pipeline import run_all
c, r = map(int, MOSAICOS.split('x'))
result, text, files = run_all(VIDEO, CAMPO, CARPETA, INICIO, FIN, CADA, MODELO, PESOS or None, (c, r), ETIQUETAS or None)
print(text)
print('\nArchivos guardados en', CARPETA)

In [ ]:
#@title 5. Ver el resultado (muestra y vídeo anotado)
from IPython.display import Image, HTML, display
from base64 import b64encode
display(Image(files['muestra'], width=1100))
if 'video' in files:
    data = b64encode(open(files['video'], 'rb').read()).decode()
    display(HTML(f'<video width="1000" controls src="data:video/mp4;base64,{data}"></video>'))

## Fotogramas para etiquetar (entrenar el modelo)
Recorre todo el vídeo (un fotograma cada 2 s), detecta, y guarda **N fotogramas variados con las cajas que ya ve el modelo** (mitad repartidos, mitad donde no vio el balón). Se crea un `.zip` en tu carpeta de Drive para subir a **Roboflow** y corregir: clases `balon` y `jugador`.

In [ ]:
#@title 6. Exportar fotogramas para Roboflow
N = 300  #@param {type:"integer"}
from voley_cv.pipeline import frames_for_labeling
n, zip_path = frames_for_labeling(VIDEO, CAMPO, CARPETA, N, 2.0, MODELO)
print(f'{n} fotogramas → {zip_path}')

## (Más adelante) Reentrenar con tus etiquetas
Cuando tengas los fotogramas corregidos en Roboflow, expórtalos en formato **COCO** (para RF-DETR) a una carpeta de Drive y ejecuta esta celda. El modelo resultante (`checkpoint_best_total.pth`) se usa en la celda 4, campo **PESOS**.

In [ ]:
#@title 7. Reentrenar
DATASET = '/content/drive/MyDrive/voley/dataset_coco'  #@param {type:"string"}
EPOCAS = 40  #@param {type:"integer"}
from voley_cv.train import train
train(DATASET, f'{CARPETA}/modelo', MODELO, EPOCAS)